# Bienvenido

Para utilizar el programa, seleccione, ejecutar todas las celdas.

**Nota: se require un token de hugging face para el modelo gemma de google que debe ser colocada en la utilidad de secretos de google colab**

In [ ]:
#############################################################

# INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS NECESARIAS PARA
# LOS MODELOS Y HERRAMIENTAS PARA LA INTERFAZ

#############################################################

# Instalación de librerías para reconocimiento de voz
# y TTS
!pip install --upgrade -q faster-whisper
!pip install coqui-tts

# Obtener clips de voz para la clonación
!git clone https://github.com/Jisf4/voice_models.git

# Importación de whisper, transformers de hugging face y xtts
from faster_whisper import WhisperModel
from transformers import pipeline
from huggingface_hub import snapshot_download
from TTS.tts.models.xtts import Xtts
from TTS.tts.configs.xtts_config import XttsConfig
from transformers.utils import logging
logging.set_verbosity_error()

# Importación de herramientas
import soundfile as sf
import time
import torch
import os
from IPython.display import display, HTML, Javascript
from google.colab import output
import base64

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

#############################################################

# PREPARACIÓN DE LOS MODELOS

#############################################################

# Selección de modelo de faster whisper para STT y uso de CUDA
model_size = "large-v3"
model_STT = WhisperModel(model_size, device="cuda", compute_type="float16")


# Selección de modelo para LLM Gemma de 2B de parámetros (ligero)
pipe = pipeline("text-generation", model="google/gemma-2b-it",
                device_map="auto")

# Descarga de modelo para TTS
MODEL_PATH = "/content/xtts"

if not os.path.exists(MODEL_PATH):
    snapshot_download(
        repo_id="coqui/XTTS-v2",
        local_dir=MODEL_PATH,
        local_dir_use_symlinks=False
    )

# Configuración de modelo xTTS
config = XttsConfig()
config.load_json("/content/xtts/config.json")

modelTTS = Xtts.init_from_config(config)
modelTTS.load_checkpoint(
    config,
    checkpoint_dir="/content/xtts/",
    use_deepspeed=False
)

modelTTS.cuda()

#############################################################

# IMPLEMENTACIÓN DE FUNCIONES

#############################################################

# STT
def STT(audio_file):
  start = time.time()
  segments, info = model_STT.transcribe(audio_file, beam_size=1) # Uso del modelo
  t_stt = time.time() - start # Medición de tiempo de inferencia
  lan = info.language # Lenguaje detectado
  text = ""
  for segment in segments:
    text = text + segment.text # Concatenación de texto detectado
  return text, lan, t_stt

## LLM
def LLM(text):
    prompt = f"Responde de forma clara y breve en máximo 3 oraciones la siguiente pregunta: {text}"
    start = time.time()
    output = pipe(
        prompt,
        max_new_tokens=60,
        max_length=200,
        temperature=0.6,
        do_sample=True
    ) # Aplicación de modelo con limitación de longitud de respuesta
    t_llm = time.time() - start # Medición de tiempo
    ans = output[0]['generated_text'].replace(prompt, "").strip() # Respuesta generada
    return ans, t_llm

# TTS
def TTS(voice, text, lan):
  gpt_cond_latent, speaker_embedding = modelTTS.get_conditioning_latents(
      audio_path=[voice]
      ) # Configuración con audio para clonación de voz
  start = time.time()
  out = modelTTS.inference(
      text=text,
      language=lan,
      gpt_cond_latent=gpt_cond_latent,
      speaker_embedding=speaker_embedding,
      #speed=0.85,
      #temperature=0.6
  ) # Uso de modelo
  t_tts=time.time()-start # Medición de tiempo
  sf.write("output.wav", out["wav"], 24000) # Guardar audio de salida
  return t_tts


#############################################################

# INTERFAZ

#############################################################

def chatbot():
    display(HTML("""

<!-- Estilos para los elementos de la interfaz -->

<style>
.container {
    font-family: Arial;
    max-width: 650px;
    margin: auto;
    text-align: center;
}

.btn {
    padding: 10px 20px;
    margin: 5px;
    border-radius: 12px;
    border: none;
    cursor: pointer;
    font-size: 15px;
}

.primary { background:#4CAF50; color:white; }
.secondary { background:#2196F3; color:white; }
.danger { background:#f44336; color:white; }

.chat {
    text-align:left;
    margin-top:20px;
}

.bubble-user {
    background:#4db0d1;
    padding:10px;
    border-radius:12px;
    color:black;
    margin:5px;
}

.bubble-bot {
    background:#fa6000;
    padding:10px;
    border-radius:12px;
    color:black;
    margin:5px;
}

.meta {
    font-size:12px;
    color:white;
    margin-bottom:10px;
}
</style>

<div class="container" id="app"></div>


<!-- Funcionamiento de los elementos de la interfaz -->
<script>
let mediaRecorder;
let audioChunks = [];
let isRecording = false;
let selectedVoice = null;

// Comprobar permisos para grabación de audio
function renderMicPermission(){
    document.getElementById("app").innerHTML = `
        <h2>Permiso de micrófono</h2>
        <p>Para usar el asistente, habilita el micrófono</p>

        <button class="btn primary" onclick="requestMicAccess()">
            Activar micrófono
        </button>

        <p id="micStatus"></p>
    `;
}

// Selección de voz
function renderVoiceSelection(){
    document.getElementById("app").innerHTML = `
        <h2>Selecciona una voz</h2>
        <button class="btn primary" onclick="selectVoice('voice_models/Josue.wav')">Josué</button>
        <button class="btn primary" onclick="selectVoice('voice_models/Geraldine.wav')">Geraldine</button>
        <button class="btn primary" onclick="selectVoice('voice_models/Italo.wav')">Italo</button>
    `;
}

// Voz seleccionada e ir a interfaz de comunicación
function selectVoice(voice){
    selectedVoice = voice;
    renderMain();
}

// Interfaz de comunicación principal
function renderMain(){
    document.getElementById("app").innerHTML = `
        <h2>Asistente de Voz</h2>
        <p>Bienvenido, presiona el botón para comenzar tu pregunta.</p>

        <button id="recordBtn" class="btn primary" onclick="toggleRecording()">
            Hablar
        </button>

        <p id="status"></p>

        <div id="chat" class="chat"></div>

        <div id="actions" style="margin-top:15px;"></div>
    `;
}


// Grabación de voz
async function toggleRecording(){
    const btn = document.getElementById("recordBtn");
    const status = document.getElementById("status");

    // Verificar si se inició la grabación
    if(!isRecording){
        const stream = await navigator.mediaDevices.getUserMedia({ audio:true });
        mediaRecorder = new MediaRecorder(stream);

        mediaRecorder.start();
        audioChunks = [];
        isRecording = true;

        // Mostrar mensajes de grabación
        btn.innerHTML = "⏹️ Detener";
        status.innerHTML = "🔴 Grabando...";

        mediaRecorder.ondataavailable = e => audioChunks.push(e.data);

    } else {

        // Verificar que se detuvo la grabación
        mediaRecorder.stop();
        isRecording = false;

        // Mostrar mensaje de procesamiento
        btn.innerHTML = "Hablar";
        status.innerHTML = "⏳ Procesando...";

        mediaRecorder.onstop = () => {
            const blob = new Blob(audioChunks);
            const reader = new FileReader();

            // Procesar audio grabado
            reader.readAsDataURL(blob);
            reader.onloadend = () => {
                google.colab.kernel.invokeFunction(
                    'notebook.process_audio',
                    [reader.result, selectedVoice],
                    {}
                );
            };
        };
    }
}


// Función para solicitar acceso al micrófono
async function requestMicAccess(){
    const status = document.getElementById("micStatus");

    try {
        const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

        status.innerHTML = "✅ Micrófono activado";
        stream.getTracks().forEach(track => track.stop());

        setTimeout(() => {
            renderVoiceSelection();
        }, 800);

    } catch (err) {
        status.innerHTML = "Debes permitir el acceso al micrófono y reiniciar la ejecución de la celda";
    }
}

// Revisar acceso al micrófono
async function checkMicPermission(){
    const result = await navigator.permissions.query({ name: 'microphone' });

    if(result.state === 'granted'){
        renderVoiceSelection();
    } else {
        renderMicPermission();
    }
}


// Mostrar burbuja de texto de usuario con métrica de tiempo
// y lenguaje detectado
function addUserBubble(text, time, lang){
    let chat = document.getElementById("chat");

    chat.innerHTML += `
        <div class="bubble-user">🧑 ${text}</div>
        <div class="meta">⏱️ ${time}s | 🌐 ${lang}</div>
    `;
    // Mostrar procesamiento finalizado
    document.getElementById("status").innerHTML = "Listo";
}


// Mostrar burbuja de texto de LLM con tiempo de generación de respuesta
// y tiempo de conversión de texto a voz
function addBotBubble(text, t_llm, t_tts){
    let chat = document.getElementById("chat");

    chat.innerHTML += `
        <div class="bubble-bot">🤖 ${text}</div>
        <div class="meta">🧠 LLM: ${t_llm}s | 🔊 TTS: ${t_tts}s</div>
    `;
    // Mostrar opciones adicionales
    showActions();
}

// Mostrar botones con opciones adicionales
function showActions(){
    document.getElementById("actions").innerHTML = `
        <button class="btn secondary" onclick="listen_audio()">Reproducir respuesta</button>
        <button class="btn primary" onclick="renderVoiceSelection()">Elegir otra voz</button>
        <button class="btn danger" onclick="endSession()">Terminar</button>
    `;
}


// REVISAR
function resetChat(){
    document.getElementById("chat").innerHTML = "";
    document.getElementById("actions").innerHTML = "";
}


// Terminar sesión
function endSession(){
    document.getElementById("app").innerHTML = "<h2>Gracias por su prueba</h2>";
    google.colab.kernel.invokeFunction('notebook.end_session', [], {});
}


// Reproducir audio
function listen_audio(){
  google.colab.kernel.invokeFunction(
                    'notebook.reproducir_audio',
                    ["output.wav"],
                    {}
                );
}

// Iniciar con comprobación de voz
checkMicPermission();
</script>
    """))



# Función para procesamientio de audio y entrega de respuesta
def process_audio(base64_audio, voice):
  header, data = base64_audio.split(',')
  audio_bytes = base64.b64decode(data)

  # Audio de usuario
  with open("input.wav", "wb") as f:
    f.write(audio_bytes)

  # STT
  texto, lang, t_stt  = STT("input.wav")

  # Mostrar consulta de usuario, tiempo de inferencia y lenguaje
  display_js(f"addUserBubble(`{texto}`, {t_stt:.2f}, `{lang}`)")

  # LLM
  respuesta, t_llm = LLM(texto)

  # TTS
  t_tts = TTS(voice, respuesta, lang)

  # Mostrar respuesta de LLM y tiempo para LLM y TTS
  display_js(f"addBotBubble(`{respuesta}`, {t_llm:.2f}, {t_tts:.2f})")

  reproducir_audio("output.wav")

# Función para reproducir el audio de salida
def reproducir_audio(path):
  with open(path, "rb") as f:
      b64 = base64.b64encode(f.read()).decode()

  display(Javascript(f"""
  if(window.currentAudio){{
        window.currentAudio.pause();
    }}

    window.currentAudio = new Audio("data:audio/wav;base64,{b64}");
    window.currentAudio.play();
    """))

# Función para mostrar elementos
def display_js(code):
  from IPython.display import Javascript
  display(Javascript(code))

# Función para terminar el programa
def end_session():
  raise SystemExit("Fin")

# Callbacks de funciones para JS
output.register_callback('notebook.process_audio', process_audio)
output.register_callback('notebook.reproducir_audio', reproducir_audio)
output.register_callback('notebook.end_session', end_session)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 70.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 3.6 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sh

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
chatbot()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>